1. Importing the librairies and dataset


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime

In [ ]:
df= pd.read_csv('ecommerce_customer_data_large.csv')

2. Data audit


In [ ]:
df.head()

,Customer ID,Purchase Date,Product Category,Product Price,Quantity,Total Purchase Amount,Payment Method,Customer Age,Returns,Customer Name,Age,Gender,Churn
0,44605,2023-05-03 21:30:02,Home,177,1,2427,PayPal,31,1.0,John Rivera,31,Female,0
1,44605,2021-05-16 13:57:44,Electronics,174,3,2448,PayPal,31,1.0,John Rivera,31,Female,0
2,44605,2020-07-13 06:16:57,Books,413,1,2345,Credit Card,31,1.0,John Rivera,31,Female,0
3,44605,2023-01-17 13:14:36,Electronics,396,3,937,Cash,31,0.0,John Rivera,31,Female,0
4,44605,2021-05-01 11:29:27,Books,259,4,2598,PayPal,31,1.0,John Rivera,31,Female,0


In [ ]:
print(f'Their are  {df.shape[0]} rows, and {df.shape[1]} columns')

Their are  250000 rows, and 13 columns


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 13 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Customer ID            250000 non-null  int64  
 1   Purchase Date          250000 non-null  object 
 2   Product Category       250000 non-null  object 
 3   Product Price          250000 non-null  int64  
 4   Quantity               250000 non-null  int64  
 5   Total Purchase Amount  250000 non-null  int64  
 6   Payment Method         250000 non-null  object 
 7   Customer Age           250000 non-null  int64  
 8   Returns                202618 non-null  float64
 9   Customer Name          250000 non-null  object 
 10  Age                    250000 non-null  int64  
 11  Gender                 250000 non-null  object 
 12  Churn                  250000 non-null  int64  
dtypes: float64(1), int64(7), object(5)
memory usage: 24.8+ MB


There seems to be wrong types


In [ ]:
# cleaning the column names
df.columns = df.columns.str.lower()
print(df.columns)

Index(['customer id', 'purchase date', 'product category', 'product price',
       'quantity', 'total purchase amount', 'payment method', 'customer age',
       'returns', 'customer name', 'age', 'gender', 'churn'],
      dtype='object')


In [ ]:
#correcting the purchase date type
df['purchase date']= pd.to_datetime(df['purchase date'])

In [ ]:
#Handling missing data
missing_data = df['returns'].isnull().sum()
missing_percent = (missing_data/len(df)*100)

print(f' There are {missing_data} values missing and in percentage it is {missing_percent} %')

 There are 47382 values missing and in percentage it is 18.9528 %


In [ ]:
df['returns'].value_counts(dropna=False)

,count
returns,
1.0,101476
0.0,101142
NaN,47382


The null values are too consequent to delete, and they  just weren't recorded, as its a big percentage i will not impute it .

In [ ]:
#Handling duplicates
df.duplicated().sum()

np.int64(0)

In [ ]:
#deleting the columns which do not serve in the analysis
df = df.drop(columns=['customer age', 'customer name'])

There are not duplicates in the dataset

In [ ]:
#checking the statitsics of the dataset
df.describe()

,customer id,purchase date,product price,quantity,total purchase amount,returns,age,churn
count,250000.000000,250000,250000.000000,250000.000000,250000.000000,202618.000000,250000.000000,250000.00000
mean,25017.632092,2021-11-06 19:26:22.286192384,254.742724,3.004936,2725.385196,0.500824,43.798276,0.20052
min,1.000000,2020-01-01 00:07:26,10.000000,1.000000,100.000000,0.000000,18.000000,0.00000
25%,12590.000000,2020-12-03 08:10:12,132.000000,2.000000,1476.000000,0.000000,30.000000,0.00000
50%,25011.000000,2021-11-06 07:39:08.500000,255.000000,3.000000,2725.000000,1.000000,44.000000,0.00000
75%,37441.250000,2022-10-11 01:21:56.249999872,377.000000,4.000000,3975.000000,1.000000,57.000000,0.00000
max,50000.000000,2023-09-13 18:42:49,500.000000,5.000000,5350.000000,1.000000,70.000000,1.00000
std,14412.515718,NaN,141.738104,1.414737,1442.576095,0.500001,15.364915,0.40039


###Insights


*   The dataset spans from 2021 to 2023
*   The age span from 18 to 70 years old
*   The other quantitative staistics seem to be normally distributed






3. Feauture engineering

In [ ]:
# for time analysis

df['year'] = df['purchase date'].dt.year
df['month'] = df['purchase date'].dt.month
df['day'] = df['purchase date'].dt.day
df['quarter'] = df['purchase date'].dt.quarter


In [ ]:
#for my fact tables

df['order id']= range(1, len(df)+1)


In [ ]:
# Revenue per unit
df['revenue per unit'] = (
    df['total purchase amount'] / df['quantity']
)

In [ ]:
#return flag
df['return_flag'] = (
    df['returns'] > 0
).astype(int)


In [ ]:
#age group
df['age group'] = pd.cut(
    df['age'],
    bins=[0,25,35,45,55,100],
    labels=['18-25','26-35','36-45','46-55','55+']
)

In [ ]:
df.head(3)

,customer id,purchase date,product category,product price,quantity,total purchase amount,payment method,returns,age,gender,churn,year,month,day,quarter,order id
0,44605,2023-05-03 21:30:02,Home,177,1,2427,PayPal,1.0,31,Female,0,2023,5,3,2,1
1,44605,2021-05-16 13:57:44,Electronics,174,3,2448,PayPal,1.0,31,Female,0,2021,5,16,2,2
2,44605,2020-07-13 06:16:57,Books,413,1,2345,Credit Card,1.0,31,Female,0,2020,7,13,3,3


4. Creating fact and dimension tables

In [ ]:
#customer dim
dim_customer = (df[['customer id', 'age','age group','gender','churn']]. drop_duplicates().reset_index(drop=True))
dim_customer.head(3)


,customer id,age,age group,gender,churn
0,44605,31,26-35,Female,0
1,13738,27,26-35,Female,0
2,33969,27,26-35,Male,0


In [ ]:
#product dim table
dim_product = (df[['product category']]. drop_duplicates().reset_index(drop=True))
dim_product['product id']= range(1,len(dim_product) + 1) #id creation for product dim table

#merging with the fact table
df = df.merge(dim_product, on='product category', how = 'left')

dim_product.head(3)


,product category,product id
0,Home,1
1,Electronics,2
2,Books,3


In [ ]:
#payment dim table

dim_payment = (df[['payment method']]. drop_duplicates().reset_index(drop=True))
dim_payment['payment id']= range(1,len(dim_payment) + 1) #id creation for payment dim table

#merging with the fact table
df = df.merge(dim_payment, on='payment method', how = 'left')

dim_payment.head(3)

,payment method,payment id
0,PayPal,1
1,Credit Card,2
2,Cash,3


In [ ]:
# date dim table
dim_date = (
    df[['purchase date', 'year', 'month', 'day', 'quarter']]
    .drop_duplicates()
    .sort_values('purchase date')
    .reset_index(drop=True)
)
dim_date['month name'] = pd.to_datetime(dim_date['purchase date']).dt.month_name()
dim_date['day name'] = pd.to_datetime(dim_date['purchase date']).dt.day_name()
dim_date['date id'] = range(1, len(dim_date) + 1)
df = df.merge(
    dim_date[['date id', 'purchase date']],
    on='purchase date',
    how='left'
)

dim_date.head(3)

,purchase date,year,month,day,quarter,month name,day name,date id
0,2020-01-01 00:07:26,2020,1,1,1,January,Wednesday,1
1,2020-01-01 00:11:40,2020,1,1,1,January,Wednesday,2
2,2020-01-01 00:15:47,2020,1,1,1,January,Wednesday,3


In [ ]:
fact_orders = df[
    [
        'order id',
        'customer id',
        'product id',
        'payment id',
        'date id',
        'quantity',
        'product price',
        'total purchase amount',
        'returns',
        'return_flag',
        'revenue per unit'
    ]
]
fact_orders.head(3)

,order id,customer id,product id,payment id,date id,quantity,product price,total purchase amount,returns,return_flag,revenue per unit
0,1,44605,1,1,225260,1,177,2427,1.0,1,2427.0
1,2,44605,2,1,92523,3,174,2448,1.0,1,816.0
2,3,44605,3,2,35808,1,413,2345,1.0,1,2345.0


In [ ]:
#re ordering the tables
dim_date = dim_date[
    ['date id', 'purchase date', 'year', 'month','month name', 'day', 'day name','quarter']
]

dim_product = dim_product[['product id', 'product category']]
dim_payment = dim_payment[['payment id', 'payment method']]

Downloading the tables

In [ ]:
# Star schema tables
dim_customer.to_csv('dim_customer.csv', index=False)
dim_product.to_csv('dim_product.csv', index=False)
dim_payment.to_csv('dim_payment.csv', index=False)
dim_date.to_csv('dim_date.csv', index=False)
fact_orders.to_csv('fact_orders.csv', index=False)